# Monthly Jobs Online Data Inspection

Inspect the monthly source file, identify available series, and
assess its suitability for a time-series forecasting task.

In [3]:
from pathlib import Path

import pandas as pd

monthly_file = Path(
    "../data/raw/jol-monthly-unadjusted-series-from-may-2007-july-2026.csv"
)

monthly_raw = pd.read_csv(
    monthly_file,
    encoding="utf-8-sig"
)

In [4]:
monthly = monthly_raw.copy()

monthly["ACTUAL_DATE"] = pd.to_datetime(
    monthly["ACTUAL_DATE"],
    format="%d/%m/%Y",
    errors="raise"
)

In [11]:
monthly.head(15)

,ACTUAL_DATE,TOTALS,ANNUAL_CHANGE,SkilledIndex,UnskilledIndex,Auckland,Wellington,North Island Other,Canterbury,South Island Other,...,Community and Personal Service Workers,Clerical and Administrative Workers,Sales Workers,Machinery Operators and Drivers,Labourers,Highly-Skilled,Skilled,Semi-Skilled,Low-Skilled,Unskilled
0,2007-05-01,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,2007-06-01,92.1,0.0,91.5,92.5,91.8,93.5,94.4,94.1,76.6,...,91.7,95.2,93.8,80.2,89.2,91.2,92.8,89.6,92.8,91.2
2,2007-07-01,95.7,0.0,95.3,94.9,96.4,95.1,95.5,98.6,84.0,...,97.3,95.9,98.7,92.3,89.5,95.1,93.7,101.7,94.1,97.4
3,2007-08-01,103.2,0.0,102.0,103.1,101.2,98.8,113.3,109.5,106.4,...,122.7,103.5,98.6,109.6,110.8,97.7,110.8,111.6,103.1,102.8
4,2007-09-01,97.0,0.0,96.7,96.4,93.9,95.1,105.7,103.9,107.4,...,111.7,90.8,96.9,104.7,103.5,91.1,112.1,98.8,95.3,100.2
5,2007-10-01,102.2,0.0,104.6,97.0,96.3,101.0,118.6,110.5,119.9,...,112.3,91.4,93.3,108.0,107.0,97.5,126.5,100.8,96.4,99.0
6,2007-11-01,101.1,0.0,106.6,90.2,96.2,98.8,118.1,102.9,122.9,...,105.8,87.1,82.5,108.4,99.2,103.2,120.7,94.9,90.4,89.3
7,2007-12-01,70.0,0.0,75.0,60.3,65.1,67.8,83.7,71.1,105.8,...,63.0,57.0,55.3,65.4,71.2,75.2,78.7,62.8,60.6,58.9
8,2008-01-01,101.8,0.0,101.9,98.6,97.1,97.0,113.3,119.2,117.4,...,122.7,94.3,89.4,109.0,119.5,104.8,91.5,107.2,96.6,105.3
9,2008-02-01,106.0,0.0,102.5,108.9,101.1,100.1,122.1,118.5,125.3,...,133.2,99.4,106.0,128.8,129.5,100.5,105.9,108.6,108.9,109.2


In [5]:
print("Shape:", monthly.shape)
print("First date:", monthly["ACTUAL_DATE"].min())
print("Last date:", monthly["ACTUAL_DATE"].max())
print("Duplicate dates:", monthly["ACTUAL_DATE"].duplicated().sum())
print(
    "Chronological order:",
    monthly["ACTUAL_DATE"].is_monotonic_increasing
)

monthly.info()

Shape: (231, 33)
First date: 2007-05-01 00:00:00
Last date: 2026-07-01 00:00:00
Duplicate dates: 0
Chronological order: True
<class 'pandas.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 33 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   ACTUAL_DATE                             231 non-null    datetime64[us]
 1   TOTALS                                  231 non-null    float64       
 2   ANNUAL_CHANGE                           231 non-null    float64       
 3   SkilledIndex                            231 non-null    float64       
 4   UnskilledIndex                          231 non-null    float64       
 5   Auckland                                231 non-null    float64       
 6   Wellington                              231 non-null    float64       
 7   North Island Other                      231 non-null    float64       
 8   Canterbury      

In [6]:
missing_counts = monthly.isna().sum()

missing_counts[missing_counts > 0]

Series([], dtype: int64)

### Check monthly continuity

In [7]:
months = pd.PeriodIndex(
    monthly["ACTUAL_DATE"].dt.to_period("M")
)

expected_months = pd.period_range(
    "2007-05",
    "2026-07",
    freq="M"
)

assert months.equals(expected_months), "Unexpected monthly timeline"

print("Monthly timeline validation passed")

Monthly timeline validation passed


### Inspect the annual-change field

In [8]:
annual_change_check = monthly[
    ["ACTUAL_DATE", "TOTALS", "ANNUAL_CHANGE"]
].copy()

annual_change_check["calculated_yoy_pct"] = (
    monthly["TOTALS"].pct_change(
        periods=12,
        fill_method=None
    ) * 100
)

annual_change_check.head(15)

,ACTUAL_DATE,TOTALS,ANNUAL_CHANGE,calculated_yoy_pct
0,2007-05-01,100.0,0.0,NaN
1,2007-06-01,92.1,0.0,NaN
2,2007-07-01,95.7,0.0,NaN
3,2007-08-01,103.2,0.0,NaN
4,2007-09-01,97.0,0.0,NaN
5,2007-10-01,102.2,0.0,NaN
6,2007-11-01,101.1,0.0,NaN
7,2007-12-01,70.0,0.0,NaN
8,2008-01-01,101.8,0.0,NaN
9,2008-02-01,106.0,0.0,NaN


In [10]:
annual_change_check.tail(5)

,ACTUAL_DATE,TOTALS,ANNUAL_CHANGE,calculated_yoy_pct
226,2026-03-01,114.8,13.8,13.776016
227,2026-04-01,95.1,6.4,6.375839
228,2026-05-01,105.1,1.1,1.057692
229,2026-06-01,101.8,13.5,13.363029
230,2026-07-01,104.0,-0.1,-0.096061


### Audit the differences

In [12]:
annual_change_check["difference_pp"] = (
    annual_change_check["ANNUAL_CHANGE"]
    - annual_change_check["calculated_yoy_pct"]
)

comparable_rows = annual_change_check.dropna(
    subset=["calculated_yoy_pct"]
)

largest_difference_rows = (
    comparable_rows["difference_pp"]
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .index
)

annual_change_check.loc[largest_difference_rows].round({
    "TOTALS": 1,
    "ANNUAL_CHANGE": 2,
    "calculated_yoy_pct": 2,
    "difference_pp": 3
})

,ACTUAL_DATE,TOTALS,ANNUAL_CHANGE,calculated_yoy_pct,difference_pp
167,2021-04-01,165.6,319.7,319.24,0.459
168,2021-05-01,187.6,164.4,164.23,0.175
76,2013-09-01,98.1,12.9,12.76,0.141
45,2011-02-01,72.9,18.1,17.96,0.139
229,2026-06-01,101.8,13.5,13.36,0.137
79,2013-12-01,64.3,18.5,18.63,-0.135
19,2008-12-01,54.2,-22.7,-22.57,-0.129
34,2010-03-01,72.4,24.1,23.97,0.127
91,2014-12-01,73.8,14.9,14.77,0.126
69,2013-02-01,83.6,1.7,1.58,0.120


### Prepare the monthly working dataset

In [13]:
monthly_clean = monthly.rename(columns={
    "ACTUAL_DATE": "Date",
    "TOTALS": "overall_index",
    "ANNUAL_CHANGE": "reported_yoy_pct"
}).copy()

monthly_clean["calculated_yoy_pct"] = (
    monthly_clean["overall_index"].pct_change(
        periods=12,
        fill_method=None
    ) * 100
)

In [14]:
actual_months = pd.PeriodIndex(
    monthly_clean["Date"].dt.to_period("M")
)

assert actual_months.equals(expected_months), (
    "Unexpected monthly timeline"
)

assert monthly_clean["overall_index"].gt(0).all(), (
    "Missing or non-positive overall index"
)

assert monthly_clean["calculated_yoy_pct"].iloc[:12].isna().all()
assert monthly_clean["calculated_yoy_pct"].iloc[12:].notna().all()

print("Monthly validation passed")
print("Shape:", monthly_clean.shape)

Monthly validation passed
Shape: (231, 34)
